# 01 · 探索性数据分析（EDA）起步

**开赛拿到数据后，头 30 分钟跑这个。** 目标不是出结论，是把数据摸熟：
有多大、什么类型、缺什么、目标长什么样。

用法：把下面的 `DATA_FILE` 改成 `data/raw/` 里的文件，逐格跑。
结论回填到 `docs/问题理解.md`，可复用的函数搬到 `src/`。

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src.config import PATHS, SEED, set_global_seed
set_global_seed(SEED)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
print('种子已固定:', SEED)

In [ ]:
# === 改这里 ===
DATA_FILE = PATHS['raw'] / 'TODO.csv'   # 赛题数据文件名
TARGET    = 'TODO'                      # 目标列名（回归/分类）
ID_COL    = None                        # 若有 ID 列，填列名

print('raw/ 下现有文件：')
for p in sorted(PATHS['raw'].rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(PATHS["raw"])}  ({p.stat().st_size/1e6:.1f} MB)')

In [ ]:
def load_any(path):
    # 按后缀自动选读取器。大文件优先 parquet/feather。
    path = Path(path)
    if path.suffix == '.csv':               return pd.read_csv(path)
    if path.suffix in ('.parquet', '.pq'):  return pd.read_parquet(path)
    if path.suffix in ('.xlsx', '.xls'):    return pd.read_excel(path)
    if path.suffix in ('.tsv', '.txt'):     return pd.read_csv(path, sep='\t')
    raise ValueError(f'未知格式: {path.suffix}')

# df = load_any(DATA_FILE)
# print(df.shape)
# df.head()

In [ ]:
def quick_profile(df, target=None):
    # 一屏摸清：类型 / 缺失 / 唯一值 / 取值样例。
    prof = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        '缺失率': df.isna().mean().round(4),
        '唯一值': df.nunique(),
        '样例': [df[c].dropna().iloc[0] if df[c].notna().any() else None
                 for c in df.columns],
    })
    print(f'形状: {df.shape[0]} 行 x {df.shape[1]} 列')
    print(f'重复行: {df.duplicated().sum()}')
    print(f'全空列: {df.isna().all().sum()}')
    if target and target in df.columns:
        print(f'\n目标列 {target}:')
        print(df[target].describe())
        print(df[target].value_counts(normalize=True).head(15))
    return prof

# quick_profile(df, TARGET)

## 下一步

- 数值列：分布、异常、是否要 log 变换 → 画直方图
- 类别列：基数、稀有类别、是否要合并 → 画条形图
- 与目标的关系：分类看分组均值差，回归看相关系数 → 排序出 top 特征
- 时间列（若有）：**务必按时间切分验证集**，别随机切（会泄漏）
- 结论写进 `docs/问题理解.md`，特征想法写进特征字典